# Compute Per-Cell Tuning Properties & Save to Zarr

For each mouse and each session, this notebook computes the following tuning metrics per cell and writes them to `ophys/drifting_gratings/session_{i}/tuning_properties` in the corresponding zarr store:

- **OSI** — Orientation Selectivity Index
- **DSI** — Direction Selectivity Index
- **pref_ori** — Preferred orientation (0–180°)
- **max_response** — Maximum mean ΔF/F across orientations
- **mean_response** — Mean ΔF/F across orientations
- **bandwidth** — Orientation tuning bandwidth (half-width at half-max, von Mises fit)
- **C50** — Semi-saturation contrast (Naka–Rushton)
- **Rmax_crf** — Maximum response from CRF fit
- **n_exponent** — Exponent from CRF fit
- **pref_TF** — Preferred temporal frequency
- **TF_max_response** — Maximum response across TFs
- **TF_lowpass_idx** — Low-pass vs band-pass index

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
from scipy.optimize import curve_fit
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from functions.tuning import (compute_osi, compute_dsi, preferred_orientation,
                               von_mises_fit, naka_rushton, compute_tuning_for_session,
                               save_tuning_to_zarr, ORIENTATIONS, CONTRASTS, TFS)

ZARR_DIR = Path('multimodal_data')
MOUSE_IDS = sorted(f.stem.split('_')[0] for f in ZARR_DIR.glob('*_multimodal_data.zarr'))
SESSIONS = ['session_1', 'session_2', 'session_3']

print(f"Mice: {MOUSE_IDS}")
print(f"Sessions: {SESSIONS}")

Mice: ['778174', '786297', '797371']
Sessions: ['session_1', 'session_2', 'session_3']


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Tuning metric functions — imported from functions.tuning
# ══════════════════════════════════════════════════════════════════════
print("Functions imported: compute_osi, compute_dsi, preferred_orientation, von_mises_fit, naka_rushton")

In [ ]:
# compute_tuning_for_session — imported from functions.tuning
print("Function imported: compute_tuning_for_session")

In [ ]:
# save_tuning_to_zarr — imported from functions.tuning

# ══════════════════════════════════════════════════════════════════════
# Main loop: mouse × session
# ══════════════════════════════════════════════════════════════════════
for mouse_id in MOUSE_IDS:
    zarr_path = str(ZARR_DIR / f'{mouse_id}_multimodal_data.zarr')
    z = zarr.open(zarr_path, mode='a')
    n_cells = z['unique_id'].shape[0]

    for sess in SESSIONS:
        gs = z[f'ophys/drifting_gratings/{sess}/stim_aligned_dff/GratingStim']

        dff = gs['dff'][:]               # (n_trials, n_timepoints, n_cells)
        time_rel = gs['time_relative'][:]

        ti_keys = list(gs['trial_info'].keys())
        trial_info = pd.DataFrame({k: gs[f'trial_info/{k}'][:] for k in ti_keys})

        tuning = compute_tuning_for_session(dff, trial_info, time_rel, n_cells)
        tp_path = f'ophys/drifting_gratings/{sess}/tuning_properties'
        try:
            del z[tp_path]  # remove existing group if present
        except KeyError:
            pass

        save_tuning_to_zarr(z, sess, tuning)

        print(f'{mouse_id} / {sess}  —  {n_cells} cells, '
              f'OSI median={tuning["OSI"].median():.3f}, '
              f'C50 non-NaN={tuning["C50"].notna().sum()}')

    # Re-consolidate metadata so new groups are discoverable
    zarr.consolidate_metadata(zarr_path)
    print(f'{mouse_id}: metadata consolidated\n')

778174 / session_1  —  614 cells, OSI median=0.303, C50 non-NaN=595
778174 / session_2  —  614 cells, OSI median=0.068, C50 non-NaN=452
778174 / session_3  —  614 cells, OSI median=0.071, C50 non-NaN=436
778174: metadata consolidated

786297 / session_1  —  1173 cells, OSI median=0.071, C50 non-NaN=1030
786297 / session_2  —  1173 cells, OSI median=0.067, C50 non-NaN=905
786297 / session_3  —  1173 cells, OSI median=0.062, C50 non-NaN=994
786297: metadata consolidated

797371 / session_1  —  1037 cells, OSI median=0.064, C50 non-NaN=756
797371 / session_2  —  1037 cells, OSI median=0.060, C50 non-NaN=639
797371 / session_3  —  1037 cells, OSI median=0.064, C50 non-NaN=529
797371: metadata consolidated



In [33]:
# ══════════════════════════════════════════════════════════════════════
# Verify saved data
# ══════════════════════════════════════════════════════════════════════
for mouse_id in MOUSE_IDS:
    z = zarr.open(str(ZARR_DIR / f'{mouse_id}_multimodal_data.zarr'), mode='r')
    for sess in SESSIONS:
        tp = z[f'ophys/drifting_gratings/{sess}/tuning_properties']
        cols = list(tp.attrs['columns'])
        n = tp.attrs['n_cells']
        print(f'{mouse_id} / {sess}: {n} cells, columns={cols}')
    print()

778174 / session_1: 614 cells, columns=['OSI', 'DSI', 'pref_ori', 'max_response', 'mean_response', 'bandwidth', 'C50', 'Rmax_crf', 'n_exponent', 'pref_TF', 'TF_max_response', 'TF_lowpass_idx']
778174 / session_2: 614 cells, columns=['OSI', 'DSI', 'pref_ori', 'max_response', 'mean_response', 'bandwidth', 'C50', 'Rmax_crf', 'n_exponent', 'pref_TF', 'TF_max_response', 'TF_lowpass_idx']
778174 / session_3: 614 cells, columns=['OSI', 'DSI', 'pref_ori', 'max_response', 'mean_response', 'bandwidth', 'C50', 'Rmax_crf', 'n_exponent', 'pref_TF', 'TF_max_response', 'TF_lowpass_idx']

786297 / session_1: 1173 cells, columns=['OSI', 'DSI', 'pref_ori', 'max_response', 'mean_response', 'bandwidth', 'C50', 'Rmax_crf', 'n_exponent', 'pref_TF', 'TF_max_response', 'TF_lowpass_idx']
786297 / session_2: 1173 cells, columns=['OSI', 'DSI', 'pref_ori', 'max_response', 'mean_response', 'bandwidth', 'C50', 'Rmax_crf', 'n_exponent', 'pref_TF', 'TF_max_response', 'TF_lowpass_idx']
786297 / session_3: 1173 cells, 